# Ingestion Pipeline — COSORA

Pipeline de ingesta: lee `.docx` y `.doc` desde Google Drive (vía montaje de acceso directo), aplica chunking inteligente, genera embeddings con MrBERT-es y guarda en ChromaDB.

## 0. Montar Google Drive

**REQUISITO:** Debes tener un "Acceso Directo" (Shortcut) de la carpeta compartida "RAG UPC Final project" en la raíz de tu "Mi Unidad" (My Drive).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q docx2txt chromadb sentence-transformers langchain langchain-text-splitters transformers
!apt-get install -y antiword > /dev/null 2>&1

import os, glob, re, unicodedata, subprocess
import docx2txt
import chromadb
from sentence_transformers import SentenceTransformer

# ── Configuración de rutas (Montaje local) ──
PROJECT_ROOT    = '/content/drive/MyDrive/RAG_UPC_Final_project'
CHROMA_PATH     = f'{PROJECT_ROOT}/chroma_db'
COLLECTION_NAME = 'cosora_chunks'
EMBED_MODEL     = 'intfloat/multilingual-e5-base'

# Crear carpeta para ChromaDB si no existe
os.makedirs(CHROMA_PATH, exist_ok=True)

if os.path.exists(PROJECT_ROOT):
    print("✅ Carpeta del proyecto encontrada.")
else:
    print(1)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 124.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are instal

## 1. Funciones de preprocesamiento

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

def clean_text(text: str) -> str:
    """Limpieza agresiva de texto extraido de .docx/.doc."""
    import unicodedata, re
    text = unicodedata.normalize("NFKC", text)
    # Caracteres de control
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    # Normalizar guiones
    text = re.sub(r'[\u00ad\u2010-\u2015\u2212\uff0d]', '-', text)
    # Normalizar comillas
    text = re.sub(r'[\u2018\u2019\u201a\u201b]', "'", text)
    text = re.sub(r'[\u201c\u201d\u201e\u201f]', '"', text)
    # Normalizar vinetas
    text = re.sub(r'[\u2022\u2023\u25cf\u25e6\u2043\u2219\u00b7]', '-', text)
    # Eliminar lineas que son solo numeros (paginas)
    text = re.sub(r'(?m)^\s*\d{1,4}\s*$', '', text)
    # Eliminar lineas repetitivas de separacion
    text = re.sub(r'([.\-_=])\1{2,}', r'\1', text)
    # Corregir espacios antes de puntuacion
    text = re.sub(r'\s+([.,;:!?)])', r'\1', text)

    # --- MEJORA 4: Limpieza agresiva ---
    # Eliminar filas de tablas rotas (lineas con 3+ pipes)
    lines = text.split('\n')
    lines = [l for l in lines if l.count('|') < 3]
    text = '\n'.join(lines)
    # Eliminar cabeceras repetitivas de actas
    text = re.sub(r'(?mi)^\s*(fecha|lugar|hora de inicio|hora de finalizacion|asistentes)\s*$', '', text)
    # Eliminar firmas y pies de pagina
    text = re.sub(r'(?mi)^\s*(firmado|firma).*$', '', text)
    # Eliminar lineas vacias multiples
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Colapsar espacios
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def chunk_text(text: str, chunk_size=300, overlap=50, min_chunk_size=50) -> list:
    """Chunking basado en Tokenizer para el modelo E5."""
    chunks = []
    if not text:
        return chunks

    tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
    splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer,
        chunk_size=chunk_size,
        chunk_overlap=overlap,
    )
    
    raw_chunks = splitter.split_text(text)
    
    for c in raw_chunks:
        c_stripped = c.strip()
        if len(c_stripped) >= min_chunk_size:
            # PREFIJO REQUERIDO PARA E5 EN DOCUMENTOS
            chunks.append(f"passage: {c_stripped}")
            
    return chunks


## 2. Lectura directa desde el sistema de archivos

In [5]:
def leer_documento(filepath: str) -> str:
    """
    Extrae texto de un archivo leyendo directamente del disco.
    Soporta .docx (vía docx2txt) y .doc (vía antiword).
    """
    try:
        if filepath.lower().endswith('.docx'):
            text = docx2txt.process(filepath)
            return text if text and text.strip() else None
            
        elif filepath.lower().endswith('.doc'):
            # antiword lee el binario .doc directamente desde la ruta de Drive
            result = subprocess.run(['antiword', filepath], capture_output=True, text=True)
            if result.returncode == 0 and result.stdout.strip():
                return result.stdout
            else:
                print(f"    [!] Error antiword: {result.stderr}")
            return None
            
        return None
    except Exception as e:
        print(f'    [!] Excepción: {e}')
        return None

## 3. Procesamiento de documentos

In [6]:
# Buscar todos los .doc y .docx en la carpeta raíz del proyecto
archivos = []
archivos.extend(glob.glob(os.path.join(PROJECT_ROOT, '*.docx')))
archivos.extend(glob.glob(os.path.join(PROJECT_ROOT, '*.doc')))

print(f'{len(archivos)} documentos encontrados\n')

documents = []

for filepath in sorted(archivos):
    filename = os.path.basename(filepath)
    doc_id   = os.path.splitext(filename)[0]
    fmt      = os.path.splitext(filename)[1]
    print(f'  [{fmt}] {filename}', end=' ')

    raw_text = leer_documento(filepath)
    if not raw_text:
        print('-> SKIP (vacío o error)')
        continue

    cleaned = clean_text(raw_text)
    chunks  = chunk_text(cleaned)
    documents.append({'doc_id': doc_id, 'chunks': chunks})
    print(f'-> {len(chunks)} chunks')

total_chunks = sum(len(d['chunks']) for d in documents)
print(f'\n{len(documents)} docs procesados, {total_chunks} chunks totales')

31 documentos encontrados

  [.doc] 244170-DOB-AVO-05-V00-A0.doc 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

-> 2 chunks
  [.doc] 244170-DOB-AVO-07-V00-A0.doc -> 3 chunks
  [.doc] 244170-DOB-AVO-09-V00.doc -> 2 chunks
  [.doc] 244170-DOB-AVO-10-V00-A0.doc -> 3 chunks
  [.doc] 244170-DOB-AVO-11-V00.doc -> 2 chunks
  [.doc] 244170-DOB-AVO-12-V00-A0.doc -> 3 chunks
  [.doc] 244170-DOB-AVO-13-V00-A0.doc -> 2 chunks
  [.doc] 244170-DOB-AVO-14-V00-A0.doc -> 3 chunks
  [.doc] 244170-DOB-AVO-16-V00-A03.doc -> 4 chunks
  [.doc] 244170-DOB-AVO-17-V00-A0.doc -> 2 chunks
  [.doc] 244170-DOB-AVO-18-V00-A0.doc -> 2 chunks
  [.doc] 244170-DOB-AVO-19-V00-A0.doc -> 2 chunks
  [.docx] 252562-DO-AVO-02-V01.docx -> 8 chunks
  [.docx] 252562-DO-AVO-03-V01.docx -> 14 chunks
  [.docx] 252562-DO-AVO-04-V01.docx -> 10 chunks
  [.docx] 252562-DO-AVO-05-V01.docx -> 8 chunks
  [.docx] 252562-DO-AVO-06-V01.docx -> 10 chunks
  [.docx] 252562-DO-AVO-07-V01.docx -> 7 chunks
  [.docx] 254275-DO-AVO-14-V07.docx -> 11 chunks
  [.docx] 254275-DO-AVO-15-V07.docx -> 12 chunks
  [.docx] 254275-DO-AVO-16-V07.docx -> 14 chunks
  [.d

## 4. Embeddings (MrBERT-es)

In [7]:
model = SentenceTransformer(EMBED_MODEL)

all_texts, all_doc_ids, all_ids = [], [], []
for doc in documents:
    for i, chunk in enumerate(doc['chunks']):
        all_texts.append(chunk)
        all_doc_ids.append(doc['doc_id'])
        all_ids.append(f"{doc['doc_id']}__c{i:04d}")

print(f'Vectorizando {len(all_texts)} chunks...')
embeddings = model.encode(all_texts, batch_size=32, show_progress_bar=True)


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Vectorizando 268 chunks...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

## 5. Guardar en ChromaDB

In [8]:
client = chromadb.PersistentClient(path=CHROMA_PATH)

# Limpieza: borrar colección anterior si existe (re-ingesta completa)
try:
    client.delete_collection(COLLECTION_NAME)
except:
    pass

collection = client.create_collection(COLLECTION_NAME)

metadatas = [
    {'doc_id': doc_id, 'chunk_id': cid}
    for doc_id, cid in zip(all_doc_ids, all_ids)
]

BATCH = 5000
for i in range(0, len(all_texts), BATCH):
    j = min(i + BATCH, len(all_texts))
    collection.add(
        embeddings=embeddings[i:j].tolist(),
        documents=all_texts[i:j],
        metadatas=metadatas[i:j],
        ids=all_ids[i:j],
    )

print(f'ChromaDB: {collection.count()} chunks guardados en {CHROMA_PATH}')

ChromaDB: 268 chunks guardados en /content/drive/MyDrive/RAG_UPC_Final_project/chroma_db


## 6. Verificación

In [9]:
print(f'Total chunks en ChromaDB: {collection.count()}\n')

for doc in documents:
    print(f'  {doc["doc_id"]}: {len(doc["chunks"])} chunks')

# Query de prueba
test_q   = 'estabilidad del talud'
test_vec = model.encode(f'query: {test_q}').tolist()
results  = collection.query(query_embeddings=[test_vec], n_results=3)

print(f'\nQuery: "{test_q}"')
print('-' * 50)
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    clean_doc = doc.replace('passage: ', '', 1)

Total chunks en ChromaDB: 268

  244170-DOB-AVO-05-V00-A0: 2 chunks
  244170-DOB-AVO-07-V00-A0: 3 chunks
  244170-DOB-AVO-09-V00: 2 chunks
  244170-DOB-AVO-10-V00-A0: 3 chunks
  244170-DOB-AVO-11-V00: 2 chunks
  244170-DOB-AVO-12-V00-A0: 3 chunks
  244170-DOB-AVO-13-V00-A0: 2 chunks
  244170-DOB-AVO-14-V00-A0: 3 chunks
  244170-DOB-AVO-16-V00-A03: 4 chunks
  244170-DOB-AVO-17-V00-A0: 2 chunks
  244170-DOB-AVO-18-V00-A0: 2 chunks
  244170-DOB-AVO-19-V00-A0: 2 chunks
  252562-DO-AVO-02-V01: 8 chunks
  252562-DO-AVO-03-V01: 14 chunks
  252562-DO-AVO-04-V01: 10 chunks
  252562-DO-AVO-05-V01: 8 chunks
  252562-DO-AVO-06-V01: 10 chunks
  252562-DO-AVO-07-V01: 7 chunks
  254275-DO-AVO-14-V07: 11 chunks
  254275-DO-AVO-15-V07: 12 chunks
  254275-DO-AVO-16-V07: 14 chunks
  254275-DO-AVO-17-V07: 16 chunks
  254275-DO-AVO-18-V07: 15 chunks
  254275-DO-AVO-19-V07: 15 chunks
  254275-DO-AVO-20-V07: 13 chunks
  254275-DO-AVO-21-V01: 14 chunks
  254275-DO-AVO-22-V01: 23 chunks
  254275-DO-AVO-23-V01: